### EDA

In [0]:
# path to the raw uploaded dataset
filtepath = "/Volumes/marathos/default/raw/TWO_CENTURIES_OF_UM_RACES.csv"

# Read the CSV-file into a Spark DataFrame
df = (spark.read
      .option("header", "true")  
      .option("inferSchema", "true")  
      .csv(filtepath))

#  display the first rows
display(df)


In [0]:
# Count all rows in the DataFrame

row_count = df.count()

row_count

In [0]:
# Count number of columns
column_count = len(df.columns)

column_count

In [0]:
# Show Spark Scheema
df.printSchema()

- Athlete year of birth, should be integer not float

In [0]:
from pyspark.sql.functions import col, count, when

# I count null values for every column
null_counts = df.select([
    count(
        when(col(column).isNull(), column)
    ).alias(column)
    for column in df.columns
])
display(null_counts)



##### Descriptive summary of numerical fieldscount nulls for each field
- Count total rows
- Count average value
- Show standard deviation
- Show min value
- Show max value 

In [0]:
# Generate descriptive statistics
summary_df = df.describe()

display(summary_df)


#### Find unique events

In [0]:
# Count unique marathon events
unique_events = df.select("Event name").distinct().count()

print(f"Number of unique events: {unique_events}")

#### How are ages distributed among the runners

In [0]:
import plotly.express as px

# Create an athlete age column 
df_age = df.withColumn(
    "athlete_age",
    col("Year of event") - col("Athlete year of birth")
)

# Aggregate ages in Spark
age_distribution = (
    df_age
    .groupBy("athlete_age")
    .count()
    .orderBy("athlete_age")
)

# Converted aggregated result to Pandas
age_pd = age_distribution.toPandas()

# Create Plotly visualization
fig = px.scatter(
    age_pd,
    x="athlete_age",
    y="count",
    title="Athlete Age Distribution"
)

# Remove gridlines
fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=False, zeroline=False)
fig.show()


In [0]:
# Show basic statistics for athlete age
df_age.select("athlete_age").describe().display()



#### Which countries are most represented?

1. Group by country
2. Count rows per country
3. Sort be descending


In [0]:
from pyspark.sql.functions import desc

# Count athletes per country
country_counts = (
    df.groupBy("Athlete country")
    .count()
    .orderBy(desc("count"))
    .limit(10)
)

display(country_counts)

#### Investigate "Athlete performance"

- I want to understand what types of format
- Unique performance values

In [0]:
df.select("Athlete performance").distinct()


In [0]:
# Show unique athlete performance exmaples
performance_examples = (
    df.select("Athlete performance")
    .distinct()
)

display(performance_examples.limit(100))

#### Athlete Performance Analysis

In [0]:
# Show unique event distance/length values
event_distance_values = (
    df.select("Event distance/length")
    .distinct()
    .orderBy("Event distance/length")
)

display(event_distance_values)

#### Analyzing number of rows that contains km, mi, h, d or unknown

In [0]:
from pyspark.sql.functions import col, lower, when

# I classify event distance/length into main unit types
df_event_units = df.withColumn(
    "event_unit_type",
    when(lower(col("Event distance/length")).contains("km"), "km")
    .when(lower(col("Event distance/length")).contains("mi"), "mi")
    .when(lower(col("Event distance/length")).contains("h"), "h")
    .when(lower(col("Event distance/length")).contains("d"), "d")
    .otherwise("Unknown")

)

# I count rows by event unit type
unit_counts = (
    df_event_units
    .groupBy("event_unit_type")
    .count()
    .orderBy("event_unit_type")
)

display(unit_counts)

## EDA Summary

The dataset contains approximately 7.5 millions rows and 13 columns

The schema inference worked well for most columns, although som columns such as athlete birth
year were inferred as double.

Several data quality issues were identified during the analysis:

- invalid athlete birth years

- unrealistic athlete ages

- missing values in athlete club and birth year columns

- inconsisten formatting in country codes

- mixed formats in event distance/length and athlete performance columns

The analysis also showed that most events are distance-based races measured in kilometers,
while a smaller portion are times events measured in hours or days.

Country distributions and athlete age distributions appeared realistic overall, although clear
outliers were detected in several columns.

The EDA provided a strong understanding of the dataset structure, data quality and business
logic before continuing to the bronza and silver layers.